# Backtesting

Imports

In [15]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import statsmodels.api as sm

### Step 1: Load cointegration and price data

In [2]:
data_folder_path = Path("data")
print(data_folder_path.exists())

True


In [3]:
coint_results = pd.read_parquet(data_folder_path / "cointegration_results.parquet")

In [4]:
print(coint_results.head())

                                 correlation   coint_t   p_value  \
formation_end ticker_0 ticker_1                                    
2016-01-01    STT      GS           0.753596 -3.368117  0.046017   
                       MS           0.739525 -4.754344  0.000454   
                       LNC          0.776851 -3.429478  0.039180   
              COP      EOG          0.781141 -4.284543  0.002706   
              KIM      O            0.762781 -4.291778  0.002637   

                                 critical_1pct  critical_5pct  critical_10pct  \
formation_end ticker_0 ticker_1                                                 
2016-01-01    STT      GS            -3.918346      -3.348304       -3.052893   
                       MS            -3.918346      -3.348304       -3.052893   
                       LNC           -3.918346      -3.348304       -3.052893   
              COP      EOG           -3.918346      -3.348304       -3.052893   
              KIM      O             

In [6]:
historical_ohlcv_df = pd.read_parquet(data_folder_path / "historical_ohlcv_df.parquet")

In [7]:
print(historical_ohlcv_df.head())

Ticker     SOLV                       KVUE                        ...  \
Price      Open High Low Close Volume Open High Low Close Volume  ...   
Date                                                              ...   
2014-01-02  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-03  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-06  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-07  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-08  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   

Ticker              R                                                 EOG  \
Price            Open       High        Low      Close  Volume       Open   
Date                                                                        
2014-01-02  51.244708  51.384891  50.683965  50.915272  453500  59.113503   
2014-01-03  51.055471  51.335842  50.838185  51.020428  303500  58.822462   
2014-01-06  51.265752  51.4479

### Step 2: Estimate relationship parameters for all pairs

In [9]:
close_prices = historical_ohlcv_df.xs("Close", level="Price", axis=1)
print(close_prices.head())

Ticker      SOLV  KVUE  SNI  LLL  WBA  EXE       CBRE         MTD  ADT  \
Date                                                                     
2014-01-02   NaN   NaN  NaN  NaN  NaN  NaN  26.340000  241.529999  NaN   
2014-01-03   NaN   NaN  NaN  NaN  NaN  NaN  26.400000  242.710007  NaN   
2014-01-06   NaN   NaN  NaN  NaN  NaN  NaN  26.250000  242.419998  NaN   
2014-01-07   NaN   NaN  NaN  NaN  NaN  NaN  26.190001  248.360001  NaN   
2014-01-08   NaN   NaN  NaN  NaN  NaN  NaN  26.190001  249.669998  NaN   

Ticker            IQV  ...        GIS   AMD        CAG       MKTX       SCHW  \
Date                   ...                                                     
2014-01-02  45.799999  ...  31.121460  3.95  16.424904  57.739132  22.172007   
2014-01-03  46.290001  ...  31.033251  4.00  16.385775  59.292953  22.240696   
2014-01-06  46.029999  ...  31.083658  4.13  16.307476  57.483101  22.163406   
2014-01-07  46.790001  ...  31.404943  4.18  16.586372  58.030479  21.931561   
2

In [25]:
def estimate_params(ticker_0, ticker_1, formation_start, formation_end):
    pair_prices = close_prices.loc[
        formation_start:formation_end,
        [ticker_0, ticker_1]
    ].dropna()

    x = pair_prices[ticker_0]
    y = pair_prices[ticker_1]

    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()

    alpha = model.params["const"]
    beta = model.params[ticker_0]

    spread = y - (alpha + beta * x)

    return alpha, beta, spread.mean(), spread.std(), spread

In [26]:
print(estimate_params("AAPL", "MSFT", "2020-01-01", "2020-12-31"))

(np.float64(93.04628733103297), np.float64(0.9765086230051582), np.float64(3.4150986825465683e-14), np.float64(7.962096255209081), Date
2020-01-02   -12.075832
2020-01-03   -13.276664
2020-01-06   -13.446754
2020-01-07   -14.483486
2020-01-08   -13.243182
                ...    
2020-12-24    -5.764317
2020-12-28    -8.131726
2020-12-29    -7.178679
2020-12-30    -8.443614
2020-12-31    -6.761628
Length: 253, dtype: float64)
